# 第22章  Softmax的死亡与复活 —— exp溢出问题

> exp(1000)=Inf->NaN。x-max(x)一行复活。Log-Sum-Exp:log_softmax稳定。Logit Bias:非法token=-inf强制JSON格式。
> 第20章(浮点)、第19章(交叉熵)、第15章(采样)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print('环境就绪')


---
## 22.1-22.6  softmax溢出+Log-Sum-Exp+Logit Bias

Naive:exp(1000)=Inf->NaN。Stable:x-max后所有exp<=1永为Inf。数学等价。Log-Sum-Exp:F.cross_entropy内部实现。
Logit Bias:非法token logit=-inf->softmax(-inf)=0->永不被选。Agent强制JSON格式。

AI连接：第29章softmax(Q@K^T/sqrt(d_k)),第35章Agent三层格式保险。


In [ ]:
import numpy as np
def sm_s(x): x=np.float64(x); x=x-np.max(x); e=np.exp(x); return e/e.sum()
x_s=np.array([2.,1.,.1]); print('safe:',sm_s(x_s).round(4))
x_d=np.array([1000.,1001.,1002.])
print('stable:',sm_s(x_d).round(4),'sum=',sm_s(x_d).sum())
xf=np.float32(x_d); n=np.exp(xf)/np.exp(xf).sum()
print('naive:',n,'<- NaN!')
# Log-Sum-Exp + log_softmax
def lse(x): x=np.float64(x); c=x.max(); return c+np.log(np.sum(np.exp(x-c)))
def lsm(x): return x-lse(x)
ls=lsm(x_d); print('log_softmax:',ls.round(4),'exp(sum)=',np.exp(ls).sum().round(4))
# Logit Bias: -inf forces probability=0
V=12; np.random.seed(42); logits=np.random.randn(V)
bias=np.full(V,-np.inf); bias[0]=0.
cp=np.exp((logits+bias)-(logits+bias).max())/np.exp((logits+bias)-(logits+bias).max()).sum()
print(f'constrained: token0={cp[0]:.4f} others<{cp[1:].max():.2e}')


---
## 习题

> 在下方新建代码单元格作答。

1.(概念)为什么softmax([1000,1001,1002])=NaN？
2.(概念)证明softmax(x)=softmax(x-c),为什么c=max(x)能解决溢出？
3.(代码)5组不同量级输入对比naive vs stable。
4.(代码)实现log_softmax,验证log_sum_exp。
5.(代码)20 token词表,Logit Bias强制输出{ -> " -> a -> " -> }。


---

> 章末钩子：Softmax不崩了。但梯度本身可能爆炸到数千——参数飞走怎么办？
预览：下一章——**梯度爆炸与梯度裁剪**。

> 提示：完成后运行 Kernel -> Restart & Run All 验证。
